#### Dealing With Different Website Layouts

In [195]:
from bs4 import BeautifulSoup
from urllib.request import urlopen

class Content:
    def __init__(self, url, title, body):
        self.url = url
        self.title = title
        self.body = body

    def print(self):
        print(f'TITLE: {self.title}')
        print(f'URL: {self.url}')
        print(f'BODY:\n {self.body}')

def scrapeCNN(url):
    bs = BeautifulSoup(urlopen(url).read(), 'html.parser') # added the "read(), 'html.parser'" code
    title = bs.find('title').text
    
    body = bs.find('div', {'class': 'article__content'})
    if body:
        body = body.get_text(strip=True)
    else:
        print("Article body not found.")
    #print('body: ')
    #print(body)
    return Content(url, title, body)

In [66]:
# This causes a HTTP Error 403
#url = 'https://www.brookings.edu/research/robotic-rulemaking'
#content = scrapeBrookings(url)
#content.print()

url = 'https://www.cnn.com/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html'
content = scrapeCNN(url)
content.print()

TITLE: Dogecoin jumps after Elon Musk replaces Twitter bird with Shiba Inu | CNN Business
URL: https://www.cnn.com/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html
BODY:
 New YorkCNN—Twitter’s traditional bird icon was booted and replaced with an image of a Shiba Inu, an apparent nod to dogecoin, the jokecryptocurrencythat CEO Elon Musk is being sued over.Musk addressed the change Monday afternoon,tweeting, “as promised” above an image of a year-old conversation in which another user suggested that Musk “just buy Twitter” and “change the bird logo to a doge.”CNN/Adobe StockRelated articleElon Musk's Twitter promised a purge of blue check marks. Instead he singled out one accountThe doge logo appeared on the site two days after Musk asked a judge to throw out a $258 billion racketeering lawsuit accusing him of running a pyramid scheme to support the dogecoin,according to Reuters.Lawyers for Musk and Tesla called the lawsuit bydogecoin investorsa “fanciful work of fiction” over

In [68]:
html2 = urlopen('https://www.cnn.com/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html')
bs2 = BeautifulSoup(html2.read(), 'html.parser')
bs2.find('div', {'class':'article__content'}).text
#bs2.find('title').text

"\n\n\nNew York\nCNN\n        \xa0—\xa0\n    \n\n\n            Twitter’s traditional bird icon was booted and replaced with an image of a Shiba Inu, an apparent nod to dogecoin, the joke cryptocurrency that CEO Elon Musk is being sued over. \n    \n\n            Musk addressed the change Monday afternoon, tweeting, “as promised” above an image of a year-old conversation in which another user suggested that Musk “just buy Twitter” and “change the bird logo to a doge.” \n    \n\n\n\n\n\n\n\n\n\n\n\nCNN/Adobe Stock\n\n\n\n\nRelated article\nElon Musk's Twitter promised a purge of blue check marks. Instead he singled out one account\n\n\n\n\n            The doge logo appeared on the site two days after Musk asked a judge to throw out a $258 billion racketeering lawsuit accusing him of running a pyramid scheme to support the dogecoin, according to Reuters.\n\n\n            Lawyers for Musk and Tesla called the lawsuit by dogecoin investors a “fanciful work of fiction” over Musk’s “innocuous

In [69]:
# This did not work
def scrapeBrookings(url):
    bs = BeautifulSoup(urlopen(url))
    title = bs.find('h1').text
    body = bs.find('div', {'class' :'post-body'}).text
    return Content(url, title, body)

In [208]:
class Content:
    """
    Common base class for all articles/pages
    """
    def __init__(self, url, title, body):
        self.url = url
        self.title = title
        self.body = body

    def print(self):
        """
        Flexible printing function controls output
        """
        print(f'URL: {self.url}')
        print(f'TITLE: {self.title}')
        print(f'BODY:\n{self.body}')
        print('\n'*3)

class Website:
    """
    Contains information about website structure
    """
    def __init__(self, name, url, titleTag, bodyTag):
        self.name = name
        self.url = url
        self.titleTag = titleTag
        self.bodyTag = bodyTag

In [209]:
from bs4 import BeautifulSoup

class Crawler:
    def getPage(url):
        try:
            html = urlopen(url)
        except Exception:
            print('Could not open page')
            return None
        return BeautifulSoup(html, 'html.parser')

    def safeGet(bs, selector):
        """
        Utility function used to get a content string from a Beautiful Soup
        object and a selector. Returns an empty string if no object
        is found for the given selector
        """
        selectedElems = bs.select(selector)
        #print(selectedElems)
        if selectedElems is not None and len(selectedElems) > 0:
            return '\n'.join([elem.get_text(strip=True) for elem in selectedElems])
        else: print('Found Nothing')
        return ''
    
    def getContent(website, path):
        """
        Extract content from a given page URL
        """
        url = website.url+path
        bs = Crawler.getPage(url)
        if bs is not None:
            title = Crawler.safeGet(bs, website.titleTag)
            body = Crawler.safeGet(bs, website.bodyTag)
            return Content(url, title, body)
        return Content(url, '', '')

In [210]:
siteData = [
    #['O\'Reilly Media', 'https://www.oreilly.com', 'h1', 'h3'],
    ['O\'Reilly Media', 'https://www.oreilly.com', 'h1', 'p'],
    ['Reuters', 'https://www.reuters.com', 'h1', 'div.ArticleBodyWrapper'], # Cannot scrape webpage
    ['Brookings', 'https://www.brookings.edu', 'title_tag', 'div'], # Cannot scrape webpage
    ['CNN', 'https://www.cnn.com', 'h1', 'div.article__content']
]
websites = []
for name, url, title, body in siteData:
    websites.append(Website(name, url, title, body))

Crawler.getContent(websites[0], '/library/view/web-scraping-with/9781491910283').print()
#Crawler.getContent(  # I get an HTTP Error 401: Forbidden
#    websites[1], '/article/us-usa-epa-pruitt-idUSKBN19W2D0').print()
#Crawler.getContent( # I get an HTTP Error 403: Forbidden
#    websites[2],
#    '/blog/techtank/2016/03/01/idea-to-retire-old-methods-of-policy-education/').print()
Crawler.getContent(
    websites[3], 
    '/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html').print()

URL: https://www.oreilly.com/library/view/web-scraping-with/9781491910283
TITLE: Web Scraping with Python
BODY:
Learn web scraping and crawling techniques to access unlimited data from any web source in any format. With this practical guide, you’ll learn how to use Python scripts and web APIs to gather and process data from thousands—or even millions—of web pages at once.
Ideal for programmers, security professionals, and web administrators familiar with Python, this book not only teaches basic web scraping mechanics, but also delves into more advanced topics, such as analyzing raw data or using scrapers for frontend website testing. Code samples are available to help you understand the concepts in practice.
Follow us
Take O'Reilly with you and learn anywhere, anytime on your phone and tablet.
View all O'Reilly videos, virtual conferences, and live events on your home TV.
Do not sell or share my personal information.
© 2025, O'Reilly Media, Inc. All trademarks and registered trademarks

#### Crawling Through Sites With Search

In [342]:
class Content:
    """ Common base class for all articles/pages"""
    def __init__(self, topic, url, title, body):
        self.topic = topic
        self.title = title
        self.body = body
        self.url = url

    def print(self):
        """
        Flexible printing function controls output
        """
        print(f'New article found for topic: {self.topic}')
        print(f'URL: {self.url}')
        print(f'TITLE: {self.title}')
        print(f'BODY:\n{self.body}')
        print('\n'*3)

In [343]:
class Website:
    """ Contains information about website structure"""
    def __init__(self, name, url, searchUrl, resultListing,
                 resultUrl, absoluteUrl, titleTag, bodyTag):
        self.name = name
        self.url = url
        self.searchUrl = searchUrl
        self.resultListing = resultListing
        self.resultUrl = resultUrl
        self.absoluteUrl = absoluteUrl
        self.titleTag = titleTag
        self.bodyTag = bodyTag

In [344]:
class Crawler:
    def __init__(self, website):
        self.site = website
        self.found = {}

    def getPage(url):
        try:
            html = urlopen(url)
        except Exception as e:
            return None
        return BeautifulSoup(html, 'html.parser')

    def safeGet(bs, selector):
        """
        Utility function used to get a content string from a
        Beautiful Soup object and a selector. Returns an empty
        string if no object is found for the given selector
        """
        selectedElems = bs.select(selector)
        #print(f'{selector}')
        if selectedElems is not None and len(selectedElems)>0:
            #print ('\n'.join([elem.get_text() for elem in selectedElems]))
            return '\n'.join([elem.get_text() for elem in selectedElems])
        return ''
    
    def getContent(self, topic, url):
        """
        Extract content from a given page URL
        """
        bs = Crawler.getPage(url)
        #print(f'getContent {bs}')
        if bs is not None:
            title = Crawler.safeGet(bs, self.site.titleTag)
            #print(title)
            #print(bs)
            body = Crawler.safeGet(bs, self.site.bodyTag)
            #print(body)
            return Content(topic, url, title, body)

    def search(self, topic):
        """
        Searches a given website for a given topic and
        records all pages found
        """
        bs = Crawler.getPage(self.site.searchUrl + topic)
        searchResults = bs.select(self.site.resultListing)
        #print(f'SearchResults: {searchResults}')
        if searchResults: 
            for result in searchResults:
                #print(f'Search:\n{result}\n')
                url = result.select(self.site.resultUrl)[0].attrs['href']
                #print(f'URL:\n{url}')
                # Check to see whether it's a relative or an absolute URL
                #if self.site.absoluteUrl:
                #    print(self.site.absoluteUrl)
                #else: print('Nothing add: {self.site.url}')
                
                url = url if self.site.absoluteUrl else self.site.url + url
                #print(f'URL New:\n{url}')
                if url not in self.found:
                    self.found[url] = self.getContent(topic, url)
                self.found[url].print()    
        else: print(f'No search results')

In [345]:
siteData = [
 # Neither website has permission to scrape
    
 #   ['Reuters', 'http://reuters.com', 'https://www.reuters.com/search/news?blob=', 'div.search-result-indiv',
 #       'h3.search-result-title a', False, 'h1', 'div.ArticleBodyWrapper'],
 #   ['Brookings', 'http://www.brookings.edu', 'https://www.brookings.edu/search/?s=',
 #       'div.article-info', 'h4.title a', True, 'h1', 'div.core-block']
    ['Wikipedia', # Name
     'https://en.Wikipedia.org', # Url
     'https://en.wikipedia.org/wiki/', # search
     'div.mw-content-ltr ul', # rListing
     'a', # rUrl <a></a>
     '', # absUrl leave empty if false
     'h1', # title tag
     'div.shortdescription' # body tag - found from the internal body of the linked webpage  
    ]
     

]
sites = []
for name, url, search, rListing, rUrl, absUrl, tt, bt in siteData:
    sites.append(Website(name, url, search, rListing, rUrl, absUrl, tt, bt))

crawlers = [Crawler(site) for site in sites]
topics = ['python']

for topic in topics:
    for crawler in crawlers:
        crawler.search(topic)

New article found for topic: python
URL: https://en.Wikipedia.org/wiki/Pythonidae
TITLE: Pythonidae
BODY:
Family of snakes




New article found for topic: python
URL: https://en.Wikipedia.org/wiki/Python_(genus)
TITLE: Python (genus)
BODY:
Genus of snakes




New article found for topic: python
URL: https://en.Wikipedia.org/wiki/Python_(programming_language)
TITLE: Python (programming language)
BODY:
General-purpose programming language




New article found for topic: python
URL: https://en.Wikipedia.org/wiki/Python_of_Aenus
TITLE: Python of Aenus
BODY:
Ancient Greek philosopher




New article found for topic: python
URL: https://en.Wikipedia.org/wiki/Python_(Efteling)
TITLE: Python (Efteling)
BODY:
Roller coaster




New article found for topic: python
URL: https://en.Wikipedia.org/wiki/Python_(automobile_maker)
TITLE: Python (automobile maker)
BODY:





New article found for topic: python
URL: https://en.Wikipedia.org/wiki/Python_(missile)
TITLE: Python (missile)
BODY:
Israeli sh

In [281]:
# Testing out CNN saerch engine scrape
# But... CNN uses javascript and
# from urllib.request import urlopen will not work

html = "https://www.cnn.com/search?q='python'"  # Replace with your target URL
bs = BeautifulSoup(urlopen(html).read(), 'html.parser')

cnn = bs.find('span', {"class": "search__results-total"})
print(f'Total Results: {cnn.get_text()}')
#bs.html.body.div.div, 
#all_links = bs.find_all('div', 
#                        {'class':'container__field-links container_list-images-with-description__field-links'}) 
                        #{'class':'container__field'}) 
#{attrs={'data-uri':'/_components/card/instances/search-0'})

#all_links
#for link in all_links:
#    href = link.get('data-open-link')
#    if href:  # Check if href attribute exists
#        print(href)

Total Results: 


#### Crawling Sites Through Links

In [188]:
class Website:

    def __init__(self, name, url, targetPattern, absoluteUrl, 
                 titleTag, bodyTag):
        self.name = name
        self.url = url
        self.targetPattern = targetPattern
        self.absoluteUrl = absoluteUrl
        self.titleTag = titleTag
        self.bodyTag = bodyTag

class Content:

    def __init__(self, url, title, body):
        self.url = url
        self.title = title
        self.body

    def print(self):
        print(f'URL: {self.url}')
        print(f'TITLE: {self.title}')
        print(f'BODY:\n{self.body}')

In [189]:
import re

class Crawler:
    def __init__(self, site):
        self.site = site
        self.visited = {}

    def getPage(url):
        try:
            html = urlopen(url)
        except Exception as e:
            print(e)
            return None
        return BeautifulSoup(html, 'html.parser')

    def safeGet(bs, selector):
        selectedElems = bs.select(selector)
        if selectedElems is not None and len(selectedElems)>0:
            return '\n'.join([elem.get_text() for elem in selectedElems])
        return ''

    def getContent(self, url):
        """
        Extract content from a given page URL
        """
        bs = Crawler.getPage(url)
        if bs is not None:
            title = Crawler.safeGet(bs, self.site.titleTag)
            body = Crawler.safeGet(bs, self.site.bodyTag)
            return Content(url, title, body)
        return Content(url, '', '')

    def crawl(self):
        """
        Get pages from website home page
        """
        bs = Crawler.getPage(self.site.url)
        targetPages = bs.findall('a', href=re.compile(self.site.targetPattern))
        for targetPage in targetPages:
            url = targetPage.attrs['href']
            url = url if self.site.absoluteUrl else f'{self.site.url}{targetPage}'
            if url not in self.visited:
                self.visited[url] = self.getContent(url)
                self.visited[url].print()

In [191]:
brookings = Website('Reuters', 'https://brookings.edu', r'\/(research|blog)\/', True, 'h1', 'div.post-body')
crawler = Crawler(brookings)
crawler.crawl()

HTTP Error 403: Forbidden


AttributeError: 'NoneType' object has no attribute 'findall'

#### Crawling Multiple Page Types

In [192]:
class Website:
    """ Common base class for all artivlese/pages"""

    def __init__(self, name, url, titleTag, bodyTag):
        self.name = name
        self.url = url
        self.titleTag = titleTag
        self.bodyTag = bodyTag

In [194]:
class Product(Website):
    """ Contains information for scraping a product page"""

    def __init__(self, name, url, titleTag, productNumber, price):
        Website.__init__(self, name, url, titletag)
        self.productNumberTag = productNumberTag
        self.priceTag = priceTag

class Article(Website):
    """ Contains information for scraping an article page"""

    def __init__(self, name, url, titleTag, bodyTag, dateTag):
        Website.__init__(self, name, url, titleTag)
        self.bodyTag = bodyTag
        self.dateTag = dateTag